# 02 — Data CleaningRaw data from an API is never perfect. This notebook fixes:- Duplicate postings- Missing salaries- Inconsistent text formatting (whitespace, casing)- Date fields stored as plain text- Missing employment type / location values- Empty job descriptions

In [1]:
import pandas as pdimport numpy as npfrom pathlib import PathBASE_DIR = Path.cwd()df = pd.read_csv(BASE_DIR / "data" / "raw" / "uk_jobs_raw.csv")print("Rows loaded:", len(df))df.info()

Rows loaded: 729<class 'pandas.core.frame.DataFrame'>RangeIndex: 729 entries, 0 to 728Columns: 16 entriesdtypes: object(13), float64(3)

In [2]:
before = len(df)df = df.drop_duplicates(subset=["job_id"])print(f"Duplicates removed: {before - len(df)}")

Duplicates removed: 0

In [3]:
string_cols = df.select_dtypes(include="object").columnsfor col in string_cols:    df[col] = df[col].astype(str).str.strip()print("Whitespace stripped from", len(string_cols), "text columns")

Whitespace stripped from 13 text columns

In [4]:
df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")print(df["date_posted"].dtype)

datetime64[ns, UTC]

In [5]:
df["employment_type"] = (    df["employment_type"]    .str.upper()    .str.replace("_", "", regex=False)    .fillna("UNKNOWN"))df["employment_type"].value_counts()

employment_typeFULLTIME     612CONTRACTOR    71PARTTIME      31INTERNSHIP    15Name: count, dtype: int64

In [6]:
df["has_salary"] = df["salary_min"].notna() & df["salary_max"].notna()print("Jobs with salary listed:", df["has_salary"].sum())print("Jobs missing salary:", (~df["has_salary"]).sum())

Jobs with salary listed: 612\nJobs missing salary: 117

In [7]:
df["location_clean"] = df["location"].str.replace(r",\\s*(England|UK|United Kingdom)$", "", regex=True)before = len(df)df = df[df["job_description"].str.len() > 20]print(f"Rows removed for empty description: {before - len(df)}")

Rows removed for empty description: 3

In [8]:
output_path = BASE_DIR / "data" / "clean" / "uk_jobs_clean.csv"output_path.parent.mkdir(parents=True, exist_ok=True)df.to_csv(output_path, index=False)print("CLEANED DATA SAVED")print("Final row count:", len(df))df.head()

CLEANED DATA SAVED\nFinal row count: 726

**Output of this notebook:** `data/clean/uk_jobs_clean.csv` — cleaned, de-duplicated, consistently formatted data, ready for feature engineering in `03_feature_engineering.ipynb`.